# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My method is K-Means clustering. My question is "what performance archetypes exist across the content inventory," which is a grouping question, not a prediction question, and there's no real label to predict (same as w02), so this has to be unsupervised. K-Means fits because it groups items by how similar they are on the features that matter, then i inspect and name whatever groups it finds. The data feeding it is the two features from my w03 contract, ctr and avg_position, built from March 2026, restricted to the ~176,738 content items that actually have real GSC coverage that month (the other ~47% aren't included because they weren't measured, not because they're inactive, same honesty limit i already wrote in w03). I won't guess a number of clusters upfront. Instead i'll try several k values, compute the silhouette score for each, and pick based on what the data actually supports, not what sounds like a nice round number of archetypes. Finally, the cluster numbers K-Means gives back (0, 1, 2...) don't mean anything on their own, so i need to look at what's actually inside each cluster, typical ctr, typical position, maybe content_type from dim_content for context, and give each one a real name a reviewer could understand, not just "cluster 2."

In [47]:
from dotenv import load_dotenv
import os
import duckdb

load_dotenv()
token = os.environ["HF_TOKEN"]  

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 0").show()
con.sql(f"DESCRIBE SELECT * FROM {CONTENT} LIMIT 0").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [48]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS clicks_sum,
        SUM(gsc_impressions) AS impressions_sum,
        AVG(gsc_avg_position) AS avg_position
    FROM {FACT}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

features['ctr'] = features['clicks_sum'] / features['impressions_sum']
features = features.dropna(subset=['ctr', 'avg_position'])
print(features.shape)
features.head()


(176738, 5)


,content_hash_id,clicks_sum,impressions_sum,avg_position,ctr
0,content_182101404f69e88f,2.0,416.0,8.842666,0.004808
1,content_e243d3dc50a86af7,0.0,6.0,23.583333,0.000000
2,content_8ed57b4607088cc1,0.0,1426.0,78.838427,0.000000
3,content_fe9bff16c2a40838,0.0,161.0,69.421792,0.000000
4,content_da09d5b2f6152356,0.0,59.0,11.778030,0.000000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

My split is grouped by client, not random rows. i shuffled the unique client ids with a fixed seed (42) and cut them in half, then put all of a client's content items into either train or test based on which half their client landed in, giving 89,851 train rows and 86,887 test rows. i'm doing it this way because content items from the same client tend to share patterns (same site, same SEO practices), so a random row split could put similar pages from the same client on both sides and make the clustering look more stable than it really is. the point of this split isn't to test accuracy like a normal model, since there's no label to be accurate against, it's a stability check: i'll fit K-Means on the train clients, then see whether content items from the test clients (clients the clustering never saw) land in similarly-shaped groups. if the archetypes hold up on clients the model never touched, that's real structure, not noise specific to this one set of pages.

In [49]:
# split by client 
content_clients = con.sql(f"""
    SELECT DISTINCT content_hash_id, client_hash_id
    FROM {FACT}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
""").df()

features = features.merge(content_clients, on='content_hash_id', how='left') # merge the content and client information
features.shape

(176738, 6)

In [ ]:
import numpy as np

unique_clients = sorted(features['client_hash_id'].unique()) # added sorted to ensure consistent ordering of clients
rng = np.random.default_rng(42) 
shuffled_clients = rng.permutation(unique_clients) #shuffle the clients to ensure randomness
split_point = len(shuffled_clients) // 2 # split the clients into two halves for training and testing

train_clients = shuffled_clients[:split_point] # select the first half of the shuffled clients for training
test_clients = shuffled_clients[split_point:] # select the second half of the shuffled clients for testing

train_data = features[features['client_hash_id'].isin(train_clients)].copy() # create a copy of the training data to avoid SettingWithCopyWarning
test_data = features[features['client_hash_id'].isin(test_clients)].copy()

print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")


Train data shape: (50510, 6)
Test data shape: (126228, 6)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [63]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [64]:
train_data = train_data[train_data['impressions_sum'] >= 30].copy()
test_data = test_data[test_data['impressions_sum'] >= 30].copy()

print(train_data.shape, test_data.shape)


(36802, 6) (88843, 6)


In [65]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_data[['ctr', 'avg_position']])

for k in [2, 3, 4, 5, 6]:
    km_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km_test.fit_predict(X_train)
    score = silhouette_score(X_train, labels, sample_size=5000, random_state=42)
    print(f"k={k}: silhouette={score:.4f}")


k=2: silhouette=0.4576
k=3: silhouette=0.5213
k=4: silhouette=0.4784
k=5: silhouette=0.4959
k=6: silhouette=0.4905


I tested k=2 through k=6 using silhouette score on the train split, after filtering out content items with fewer than 30 total impressions. That volume floor was necessary because a first attempt without it gave a suspiciously perfect k=2 score of 0.94, driven entirely by about 279 pages with only 1-2 impressions where a single lucky click inflated their ctr to near 100%, not a real archetype. I also found and fixed a reproducibility bug where the client split wasn't actually stable across runs, because the warehouse query didn't guarantee row order, which meant "sorting" the client list before shuffling was needed to make the fixed seed actually mean something. On the cleaned, reproducible data, k=3 had the clearest silhouette score at 0.5213, ahead of k=2 (0.4576), k=4 (0.4784), k=5 (0.4959), and k=6 (0.4905). Compared to a dummy baseline of random cluster assignment, which scored -0.0078, essentially zero, confirming random groups have no real structure, k=3 is a genuine improvement. More importantly, applying the exact scaler and cluster centers fit on train to the test clients, clients the model never saw during fitting, gave a silhouette of 0.6357, even stronger than train, and the three cluster profiles matched almost exactly in shape between train and test. That stability on unseen clients is the real evidence these are genuine archetypes, not noise specific to one set of pages.

In [67]:
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
train_data['cluster_k3'] = km3.fit_predict(X_train)

print("k=3 :")
print(train_data.groupby('cluster_k3')[['ctr', 'avg_position']].agg(['mean', 'count']))



k=3 :
                 ctr        avg_position       
                mean  count         mean  count
cluster_k3                                     
0           0.002206  24655    10.249772  24655
1           0.001082   9422    35.842892   9422
2           0.020530   2725     9.700135   2725


In [68]:
rng2 = np.random.default_rng(42)
random_labels = rng2.integers(0, 3, size=X_train.shape[0])
dummy_score = silhouette_score(X_train, random_labels, sample_size=5000, random_state=42)
print(f"dummy baseline (random k=3) silhouette={dummy_score:.4f}")

dummy baseline (random k=3) silhouette=-0.0078


In [69]:
X_test = scaler.transform(test_data[['ctr', 'avg_position']])
test_labels = km3.predict(X_test)
test_score = silhouette_score(X_test, test_labels, sample_size=5000, random_state=42)
print(f"k=3 silhouette on test (clients the model has not seen) = {test_score:.4f}")

test_data_labeled = test_data.copy()
test_data_labeled['cluster_k3'] = test_labels
print(test_data_labeled.groupby('cluster_k3')[['ctr','avg_position']].agg(['mean', 'count']))

k=3 silhouette on test (clients the model has not seen) = 0.6357
                 ctr        avg_position       
                mean  count         mean  count
cluster_k3                                     
0           0.001985  66881     8.201261  66881
1           0.000586  18669    43.587680  18669
2           0.021409   3293     9.837661   3293


In [71]:
import pandas as pd
train_silhouette = silhouette_score(X_train, train_data['cluster_k3'], sample_size=5000, random_state=42)

comparison = pd.DataFrame({
    'method': ['dummy baseline (random)', 'K-Means k=3 (train)', 'K-Means k=3 (test, unseen clients)'],
    'silhoutte': [dummy_score, train_silhouette, test_score],
    'n_rows': [X_train.shape[0], X_train.shape[0], X_test.shape[0]],
})
comparison

,method,silhoutte,n_rows
0,dummy baseline (random),-0.007841,36802
1,K-Means k=3 (train),0.521325,36802
2,"K-Means k=3 (test, unseen clients)",0.635669,88843


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [72]:
# pages in "Underperforming" that are closest to Champion-level CTR
test_data_labeled[test_data_labeled['cluster_k3'] == 0].sort_values('ctr', ascending=False).head(3)[['content_hash_id', 'ctr', 'avg_position']]

,content_hash_id,ctr,avg_position
23486,content_fd4dc0be87f33ae6,0.011364,13.678937
86763,content_91a87b7f446853bd,0.011364,19.607955
34818,content_01d5f41255731a13,0.011364,21.889205


In [73]:
# pages in "Buried" that have the best position, closest to escaping that cluster
test_data_labeled[test_data_labeled['cluster_k3'] == 1].sort_values('avg_position').head(3)[['content_hash_id', 'ctr', 'avg_position']]


,content_hash_id,ctr,avg_position
145838,content_0912dac84f5b70ca,0.0,22.698333
103344,content_bb1ff7691eb61ca5,0.0,22.699383
10093,content_89bc986be0f9fc2f,0.0,22.700758


I named the three clusters based on what's actually inside them: cluster 2 is Champions (good position, good ctr, converting visibility into real clicks), cluster 0 is Underperforming Good Position (similar position to Champions but ctr is roughly 10x worse), and cluster 1 is Buried and Ignored (poor position and poor ctr). What the clustering leans on: ctr is what separates Champions from Underperforming, since both sit at almost the same avg_position (around 8-10) but differ hugely in ctr. avg_position is what separates Buried from the other two, since its average position (around 44) is far worse than either of the other clusters. Borderline cases show this isn't always clean. The three Underperforming pages closest to Champion-level ctr all sit at 1.14%, better than that cluster's 0.20% average, but still well below Champions' 2.05%, so they stay grouped as Underperforming. More telling: the three Buried pages with the best position (around 22.7, much better than that cluster's 43.6 average) still have zero clicks. Good position alone isn't enough to escape Buried, these pages prove a page can rank reasonably and still get completely ignored, which means for pages like these, chasing a better ranking wouldn't help on its own, they need something like a better title or meta description to actually earn clicks once they're visible.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.